# Load UniProt official registry

In [ ]:
import json
from pathlib import Path
import requests

In [ ]:
params = {"format": "json", "query": "*", "size": 500}

In [ ]:
response = requests.get("https://rest.uniprot.org/database/search", params=params)

In [ ]:
response.raise_for_status()

In [ ]:
registry_data = response.json()

print(type(registry_data))
print(registry_data.keys())

<class 'dict'>
dict_keys(['results'])


In [ ]:
uniprot_official_name_set = {
    entry["name"].strip().lower() for entry in registry_data["results"] if isinstance(entry, dict) and entry.get("name")
}

print("Official UniProt name count:", len(uniprot_official_name_set))

Official UniProt name count: 185


In [ ]:
uniprot_official_set = {
    entry["abbrev"].strip().lower()
    for entry in registry_data["results"]
    if isinstance(entry, dict) and entry.get("abbrev")
}

print("Official UniProt abbrev count:", len(uniprot_official_set))
print("Sample:", sorted(list(uniprot_official_set))[:20])

Official UniProt abbrev count: 185
Sample: ['abcd', 'agora', 'agr', 'allergome', 'alphafolddb', 'antibodypedia', 'antifam', 'arachnoserver', 'araport', 'bgee', 'bindingdb', 'biocyc', 'biogrid', 'biogrid-orcs', 'biomuta', 'bmrb', 'brenda', 'carbonyldb', 'card', 'cazy']


## Load BERDL prefixes.txt

In [ ]:
BERDL_PREFIXES = Path("prefixes.txt")

In [ ]:
berdl_set = set()

In [ ]:
with BERDL_PREFIXES.open() as f:
    for line in f:
        berdl_set.add(line.strip().lower())

print("BERDL idmapping prefixes:", len(berdl_set))

BERDL idmapping prefixes: 103


## Load parquet prefixes 

In [ ]:
from pyspark.sql.functions import lower, col

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.appName("PrefixExploration").getOrCreate()

In [ ]:
df = spark.read.parquet("part-00000-0a0d0261-1fee-477d-90d8-1df048058fbf-c000.snappy.parquet")

In [ ]:
df.printSchema()

root
 |-- entity_id: string (nullable = true)
 |-- db: string (nullable = true)
 |-- xref: string (nullable = true)
 |-- description: string (nullable = true)
 |-- _dlt_load_id: string (nullable = true)
 |-- _dlt_id: string (nullable = true)
 |-- relationship: string (nullable = true)



In [ ]:
parquet_set = {row["db"] for row in df.select(lower(col("db")).alias("db")).distinct().collect()}

print("Parquet prefixes:", len(parquet_set))

Parquet prefixes: 82


In [ ]:
## Prefixes in parquet but not in UniProt official list
parquet_not_in_uniprot = parquet_set - uniprot_official_set

In [ ]:
print(len(parquet_not_in_uniprot))
print(sorted(list(parquet_not_in_uniprot)))

3
['ec', 'ncbitaxon', 'uniprot']


### Interpretation

These are not true registry gaps:

- **ec** – Represents EC numbers. 
- **ncbitaxon** – A naming variation of NCBI Taxonomy.
- **uniprot** – UniProt itself is not listed as an external cross-reference database.

Conclusion: No external databases detected


In [ ]:
## Prefixes in BERDL idmapping but not in UniProt official list
berdl_not_in_uniprot = berdl_set - uniprot_official_set

In [ ]:
print(len(berdl_not_in_uniprot))
print(sorted(list(berdl_not_in_uniprot)))

25
['crc64', 'embl-cds', 'ensembl_pro', 'ensembl_trs', 'ensemblgenome', 'ensemblgenome_pro', 'ensemblgenome_trs', 'gene_name', 'gene_orderedlocusname', 'gene_orfname', 'gene_synonym', 'gi', 'ncbi_taxid', 'nextprot', 'pharmgkb', 'refseq_nt', 'treefam', 'uniparc', 'uniprotkb-id', 'uniref100', 'uniref50', 'uniref90', 'wbparasite_trs_pro', 'wormbase_pro', 'wormbase_trs']


### Classification of Differences

### Classification of BERDL Prefixes Not Present in UniProt Official Cross-Reference Registry

The following prefixes appear in the BERDL idmapping-derived set but are not listed in the UniProt official cross-reference registry.

They fall into several categories:

    1.	Internal UniProt metadata
	2.	Subtype mappings 
	3.	External biological databases
	4.	Deprecated or taxonomy identifiers
	5.	UniProt-derived resources


In [ ]:
registry_data["results"][:1]

[{'name': 'ABCD curated depository of sequenced antibodies',
  'id': 'DB-0236',
  'abbrev': 'ABCD',
  'linkType': 'Explicit',
  'servers': ['https://web.expasy.org/abcd'],
  'dbUrl': 'https://web.expasy.org/cgi-bin/abcd/search_abcd.pl?input=%u',
  'category': 'Protocols and materials databases',
  'statistics': {'reviewedProteinCount': 3196, 'unreviewedProteinCount': 619}}]

In [ ]:
from collections import defaultdict

In [ ]:
## create a dictionary with empty lists

classification = defaultdict(list)

In [ ]:
SUBTYPE_MAPPING = {
    "embl-cds",  # EMBL CDS subtype
    "refseq_nt",  # RefSeq nucleotide subtype
}

for p in sorted(berdl_not_in_uniprot):
    # internal annotation fields
    if p.startswith("gene_") or p in {"crc64", "uniprotkb-id"}:
        classification["internal_metadata"].append(p)

    # UniProt derived databases
    elif p.startswith("uniref") or p == "uniparc":
        classification["uniprot_derived_db"].append(p)

    # deprecated identifiers
    elif p in {"gi"}:
        classification["deprecated_identifier"].append(p)

    # taxonomy identifiers
    elif p in {"ncbi_taxid"}:
        classification["taxonomy_identifier"].append(p)

    # subtype-specific identifiers
    elif p in SUBTYPE_MAPPING or any(token in p for token in ["_pro", "_trs"]):
        classification["subtype_mapping"].append(p)

    # external database candidate
    else:
        classification["external_database_candidate"].append(p)

In [ ]:
for k, v in classification.items():
    print(f"\n{k} ({len(v)}):")
    print(sorted(v))


internal_metadata (6):
['crc64', 'gene_name', 'gene_orderedlocusname', 'gene_orfname', 'gene_synonym', 'uniprotkb-id']

subtype_mapping (9):
['embl-cds', 'ensembl_pro', 'ensembl_trs', 'ensemblgenome_pro', 'ensemblgenome_trs', 'refseq_nt', 'wbparasite_trs_pro', 'wormbase_pro', 'wormbase_trs']

external_database_candidate (4):
['ensemblgenome', 'nextprot', 'pharmgkb', 'treefam']

deprecated_identifier (1):
['gi']

taxonomy_identifier (1):
['ncbi_taxid']

uniprot_derived_db (4):
['uniparc', 'uniref100', 'uniref50', 'uniref90']


In [ ]:
external_candidates = classification["external_database_candidate"]

In [ ]:
for p in external_candidates:
    in_name = p in uniprot_official_name_set
    in_abbrev = p in uniprot_official_set
    count = df.filter(lower(col("db")) == p).count()
    print(f"{p:20} | name={in_name} | abbrev={in_abbrev}")
    print(f"{p:20} | parquet_rows={count}")

ensemblgenome        | name=False | abbrev=False
ensemblgenome        | parquet_rows=0
nextprot             | name=False | abbrev=False
nextprot             | parquet_rows=0
pharmgkb             | name=False | abbrev=False
pharmgkb             | parquet_rows=0
treefam              | name=False | abbrev=False
treefam              | parquet_rows=0


Some prefixes classified as external database candidates (e.g., nextprot, pharmgkb, treefam) do not currently appear in the BERDL parquet dataset.

This indicates that while these namespaces correspond to real biological databases, they are not used in the current dataset snapshot. They remain classified as external databases based on their semantic meaning rather than dataset usage.

### A. UniProt annotation metadata 
- crc64
- gene_name
- gene_orderedlocusname
- gene_orfname
- gene_synonym
- uniprotkb-id

These fields represent internal UniProt annotations rather than cross-references to external databases. Examples include gene name annotations and sequence checksums maintained directly within UniProt records.


### B. UniProt derived databases 
- uniparc 
- uniref100
- uniref50
- uniref90

These are UniProt internal resources, no need remapping. 


### C. Internal NCBI identifiers
- gi
- ncbi_taxid
  
ncbi_taxid is taxonomy identifier,
gi used by NCBI but has been officially deprecated.



### D. Database subtype mappings 
- embl-cds
- refseq_nt
- ensembl_pro
- ensembl_trs
- ensemblgenome_pro
- ensemblgenome_trs
- wormbase_pro
- wormbase_trs
- wbparasite_trs_pro

#### patterns:
	•	_pro → protein identifiers
	•	_trs → transcript identifiers
	•	_cds → coding sequence identifiers
	•	_nt → nucleotide accessions

These indicate the identifier type within a parent database. Need normalize to parent database prefix.


### E. External database 

- ensemblgenome
- nextprot
- pharmgkb
- treefam


#### examples:

	•	EnsemblGenome – genome annotation database
	•	NextProt – human protein knowledgebase
	•	PharmGKB – pharmacogenomics database
	•	TreeFam – phylogenetic gene family database

In [ ]:
pip install bioregistry

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import bioregistry as br

In [ ]:
print(br)

<module 'bioregistry' from '/home/user233/.local/lib/python3.13/site-packages/bioregistry/__init__.py'>


In [ ]:
prefixes = sorted(berdl_not_in_uniprot)

In [ ]:
results = []

for p in prefixes:
    resource = br.get_resource(p)
    normalized = br.normalize_prefix(p)

    results.append({"prefix": p, "bioregistry_found": resource is not None, "normalized": normalized})

for r in results:
    print(r)

{'prefix': 'crc64', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'embl-cds', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'ensembl_pro', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'ensembl_trs', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'ensemblgenome', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'ensemblgenome_pro', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'ensemblgenome_trs', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'gene_name', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'gene_orderedlocusname', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'gene_orfname', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'gene_synonym', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'gi', 'bioregistry_found': False, 'normalized': None}
{'prefix': 'ncbi_taxid', 'bioregistry_found': True, 'normalized': 'ncbitaxon'}
{'prefix': 'nextprot', 'bio

## conclusion 

The Bioregistry package partially support prefix remapping, but it is not sufficient as a solution for the UniProt / BERDL prefix governance workflow.

### Bioregistry package effective for: 

- Canonical prefix normalization
- Synonym resolution (e.g., ncbi_taxid → ncbitaxon)
- Validation of recognized external biological databases

### Bioregistry does not cover: 

- Subtype-specific identifiers (e.g., ensembl_pro, refseq_nt)
- UniProt internal metadata fields (e.g., gene_name, crc64)
- UniProt-derived internal resources (e.g., uniref100)
- Deprecated identifiers, requires manual handling or exclusion (e.g., gi)

In [ ]:
classification = dict(classification)

In [ ]:
INTERNAL_PREFIXES = set(classification.get("internal_metadata", []))

In [ ]:
## subtype has feature, xxx_pro/xxx_trs/xxx_nt/xxx_cds

SUBTYPE_TOKENS = {"pro", "trs", "nt", "cds"}

In [ ]:
## deducing the parent from the token


def infer_parent_prefix(prefix: str) -> str:
    tokens = prefix.replace("-", "_").split("_")
    tokens = [t for t in tokens if t not in SUBTYPE_TOKENS]
    return "_".join(tokens)

In [ ]:
SUBTYPE_RULES = {p: infer_parent_prefix(p) for p in classification.get("subtype_mapping", [])}

In [ ]:
SUBTYPE_RULES

{'embl-cds': 'embl',
 'ensembl_pro': 'ensembl',
 'ensembl_trs': 'ensembl',
 'ensemblgenome_pro': 'ensemblgenome',
 'ensemblgenome_trs': 'ensemblgenome',
 'refseq_nt': 'refseq',
 'wbparasite_trs_pro': 'wbparasite',
 'wormbase_pro': 'wormbase',
 'wormbase_trs': 'wormbase'}

In [ ]:
## INTERNAL_PREFIXES:
## if prefix.startswith("gene_")
## if prefix in {"crc64"}
## if prefix.endswith("-id")

INTERNAL_KEYWORDS = {
    "crc64",  ## only need UniProt checksum, not namespace
}


def is_internal_prefix(prefix: str) -> bool:
    prefix = prefix.lower()

    if prefix.startswith("gene_"):
        return True

    if prefix in INTERNAL_KEYWORDS:
        return True

    if prefix.endswith("-id"):
        return True

    return False

In [ ]:
INTERNAL_PREFIXES = {p for p in berdl_set if is_internal_prefix(p)}

In [ ]:
INTERNAL_PREFIXES

{'crc64',
 'gene_name',
 'gene_orderedlocusname',
 'gene_orfname',
 'gene_synonym',
 'uniprotkb-id'}

In [ ]:
DEPRECATED_PREFIXES = set(classification.get("deprecated_identifier", []))

In [ ]:
def remap_prefix(prefix: str) -> dict:
    prefix = prefix.lower()

    if is_internal_prefix(prefix):
        return {"original": prefix, "canonical": None, "source": "internal"}

    if prefix in DEPRECATED_PREFIXES:
        return {"original": prefix, "canonical": None, "source": "deprecated"}

    if prefix in SUBTYPE_RULES:
        return {"original": prefix, "canonical": SUBTYPE_RULES[prefix], "source": "subtype"}

    normalized = br.normalize_prefix(prefix)
    if normalized:
        return {"original": prefix, "canonical": normalized, "source": "bioregistry"}

    return {"original": prefix, "canonical": None, "source": "unresolved"}

In [ ]:
results = [remap_prefix(p) for p in sorted(berdl_set)]
df = pd.DataFrame(results)
df["source"].value_counts()

source
bioregistry    56
unresolved     31
subtype         9
internal        6
deprecated      1
Name: count, dtype: int64

In [ ]:
df[df["source"] == "unresolved"].sort_values("original")
df = df.drop(columns=["canonical"])

,original,canonical,source
5,biomuta,NaN,unresolved
9,chitars,NaN,unresolved
10,collectf,NaN,unresolved
13,cptac,NaN,unresolved
18,dmdm,NaN,unresolved
19,dnasu,NaN,unresolved
23,embl,NaN,unresolved
29,ensemblgenome,NaN,unresolved
32,esther,NaN,unresolved
33,euhcvdb,NaN,unresolved


#### These are UniProt cross-reference databases. 

Uniprot cluster resources not in bioregistry. 

In [ ]:
## The mappings that already made when prefixes are existing in Bioregistry

print(br.normalize_prefix("tair"))
print(br.normalize_prefix("patric"))
print(br.normalize_prefix("oma"))
print(br.normalize_prefix("merops"))

None
None
None
None


These prefixes are for external biological databases not covered by the Bioregistry.

### Final Evaluation

Bioregistry was evaluated for prefix remapping.
- It directly recognizes 56 out of 103 observed prefixes;
- It correctly normalizes prefix variants; 
- 31 prefixes are not covered by Bioregistry; these correspond mainly to UniProt-specific resources.

After incorporating subtype rules, internal metadata handling and deprecated identifiers, ~70% of prefixes can be governed.
